# 05 DAG and Query Plans

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Read Spark query plans confidently: the four plan phases (parsed, analyzed, optimized, physical), the <code>explain</code> modes, the most common physical operators, and how the physical plan becomes a <b>DAG</b> of stages.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
<code>explain()</code> is the only way to know what Spark will <b>actually</b> do with your code: whether a filter was pushed down, which join strategy was chosen, how many shuffles there are. It's the first tool in every performance investigation, and reading a plan out loud is a strong interview signal.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 From code to execution</b><br>
<code>`</code><br>your code / SQL<br>     │  parse<br>     ▼<br>Parsed (unresolved) logical plan     names not checked yet ('id, 'country)<br>     │  analyze (catalog, types)<br>     ▼<br>Analyzed logical plan                names resolved (id#12L), types known<br>     │  optimize (Catalyst rules)<br>     ▼<br>Optimized logical plan               pushdown, pruning, constant folding, ...<br>     │  plan (strategies, cost)<br>     ▼<br>Physical plan                        concrete operators: FileScan, HashAggregate, SortMergeJoin, Exchange<br>     │  AQE may re-plan at runtime<br>     ▼<br>DAG of stages (split at Exchange) -> tasks<br><code>`</code><br><b>Read physical plans bottom-up</b>: data starts at the scans at the bottom and flows up to the result.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A report query takes 20 minutes. <code>explain("formatted")</code> shows a <code>CartesianProduct</code> instead of a join: a join condition used <code>!=</code> instead of <code>=</code>. In another case the plan shows <code>PushedFilters: []</code> on a 3 TB scan because the date filter was wrapped in a UDF. Both problems are found in seconds by reading the plan, before touching any cluster setting.
</div>

## explain modes

| Call | Shows |
|---|---|
| `df.explain()` | Physical plan only |
| `df.explain(True)` / `explain("extended")` | Parsed, analyzed, optimized and physical plans |
| `df.explain("formatted")` | Compact operator tree plus numbered details per operator (best for reading) |
| `df.explain("codegen")` | The generated Java code for each codegen stage |
| `df.explain("cost")` | Optimized plan with size statistics when available |
| SQL: `EXPLAIN EXTENDED ...`, `EXPLAIN FORMATTED ...`, `EXPLAIN CODEGEN ...`, `EXPLAIN COST ...` | The same from SQL |

## Common physical operators

| Operator | Meaning |
|---|---|
| `FileScan` / `Scan` / `PhotonScan` | Read data. Check `ReadSchema`, `PushedFilters`, `PartitionFilters` |
| `Filter`, `Project` | Row filter, column selection or computation |
| `HashAggregate` (partial / final) | Aggregation. Appears twice around an `Exchange` |
| `Exchange hashpartitioning` / `rangepartitioning` / `RoundRobinPartitioning` / `SinglePartition` | Shuffle: stage boundary |
| `BroadcastExchange` + `BroadcastHashJoin` | Small side sent to all executors |
| `Sort` + `SortMergeJoin` | Both sides shuffled and sorted, then merged |
| `Window` | Window function evaluation |
| `TakeOrderedAndProject` | Top-N (`orderBy` + `limit`) |
| `*(n)` prefix | Operators fused in whole-stage code generation stage n |
| `AdaptiveSparkPlan isFinalPlan=false` | AQE will re-optimize at runtime |

## Setup: a realistic query

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates orders and customers, writes orders to Parquet so the plan has a real file scan, and defines a query with a filter, a join, an aggregation and a sort.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The query result: revenue per country for 2024, sorted.
</div>

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spark_basics")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.spark_basics.raw_files")
path = "/Volumes/workspace/spark_basics/raw_files/lesson02_05/orders"

In [0]:
from pyspark.sql import functions as F

(
    spark.range(200_000)
    .select(
        F.col("id").alias("order_id"),
        (F.col("id") % 500).alias("customer_id"),
        F.expr("date_add(date'2023-07-01', cast(id % 365 AS INT))").alias("order_date"),
        (F.col("id") % 1000 / 10).alias("amount"),
        F.lit("note").alias("comment"),
    )
    .write.mode("overwrite").parquet(path)
)
orders = spark.read.parquet(path)
customers = spark.range(500).select(
    F.col("id").alias("customer_id"),
    F.element_at(F.array(F.lit("IN"), F.lit("UK"), F.lit("AE")), (F.col("id") % 3 + 1).cast("int")).alias("country"),
)

query = (
    orders.join(customers, "customer_id")
    .filter(F.year("order_date") == 2024)
    .groupBy("country")
    .agg(F.round(F.sum("amount"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
query.show()

## 1. The four plan phases

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints all four plans with <code>explain("extended")</code>.<br><br>
<b>Why it matters</b><br>Each phase answers a different question. <b>Parsed</b>: what did you write? <b>Analyzed</b>: did every name resolve, and to which types? <b>Optimized</b>: what did Catalyst change? <b>Physical</b>: how will it run?
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b>
<ul><li>Parsed plan: unresolved names with quotes (<code>'country</code>)</li><li>Analyzed plan: resolved names with IDs (<code>country#12</code>) and types</li><li>Optimized plan: the <code>year(order_date) = 2024</code> filter has moved below the join, and only the needed columns remain</li><li>Physical plan: concrete operators, with <code>comment</code> gone from the scan</li></ul>
</div>

In [0]:
query.explain("extended")

## 2. The formatted plan: the best way to read

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the same query with <code>explain("formatted")</code>.<br><br>
<b>Why it matters</b><br>The formatted plan gives a short tree at the top, then numbered details for each operator. Read the tree bottom-up, then look up the details of the interesting operators (scans, exchanges, joins).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A tree with a <code>Scan parquet</code> (and its <code>ReadSchema</code> without <code>comment</code>), a <code>BroadcastHashJoin</code> (customers is small), two <code>HashAggregate</code> nodes around an <code>Exchange</code>, and a final <code>Sort</code> after a <code>rangepartitioning</code> exchange.
</div>

In [0]:
query.explain("formatted")

## 3. Read the scan: pushdown and pruning

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Extracts the scan line from the plan to check what is read from storage.<br><br>
<b>Why it matters</b><br>The scan is where most time goes on big data. Three things to check every time:
<ul><li><code>ReadSchema</code>: only the needed columns?</li><li><code>PushedFilters</code>: are filters reaching the reader?</li><li><code>PartitionFilters</code>: is partition pruning happening (on partitioned tables)?</li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>ReadSchema</code> with <code>customer_id</code>, <code>order_date</code> and <code>amount</code> only. <code>PushedFilters</code> shows <code>IsNotNull</code> filters. The <code>year(...)</code> filter is not pushed because it wraps the column in a function (compare lesson 01_spark_basics/15).
</div>

In [0]:
import io, contextlib, re

def plan_text(df, mode="simple"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

# In the formatted plan, each scan property is on its own line.
for line in plan_text(query, "formatted").splitlines():
    if line.startswith(("ReadSchema", "PushedFilters", "PartitionFilters")):
        print(line)

## 4. Compare a better query

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Rewrites the date filter as a range on the raw column and compares the pushed filters.<br><br>
<b>Why it matters</b><br>The plan tells you when a small change in code (a range instead of a function) lets Spark skip data. Use plans to compare alternatives, not guesses.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The range version shows <code>GreaterThanOrEqual(order_date, 2024-01-01)</code> and <code>LessThan(order_date, 2025-01-01)</code> in <code>PushedFilters</code>.
</div>

In [0]:
better = (
    orders.filter((F.col("order_date") >= "2024-01-01") & (F.col("order_date") < "2025-01-01"))
    .join(customers, "customer_id")
    .groupBy("country").agg(F.round(F.sum("amount"), 2).alias("revenue"))
)
print([l for l in plan_text(better, "formatted").splitlines() if l.startswith("PushedFilters")])
print("same result:", sorted(better.collect()) == sorted(query.collect()))

## 5. Plans in SQL, and the cost mode

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs <code>EXPLAIN FORMATTED</code> from SQL, and prints the <code>cost</code> mode, which adds size estimates.<br><br>
<b>Why it matters</b><br>SQL users read plans the same way. Size estimates (<code>sizeInBytes</code>, <code>rowCount</code> when table statistics exist) are what the optimizer uses to choose a broadcast join.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The SQL formatted plan, then an optimized plan annotated with <code>Statistics(sizeInBytes=...)</code>.
</div>

In [0]:
orders.createOrReplaceTempView("orders_v")
customers.createOrReplaceTempView("customers_v")
print(spark.sql("""
    EXPLAIN FORMATTED
    SELECT c.country, sum(o.amount) FROM orders_v o JOIN customers_v c USING (customer_id) GROUP BY c.country
""").first()[0][:1500])

query.explain("cost")

## 6. From physical plan to DAG of stages

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Annotates the physical plan with stage numbers by splitting at each <code>Exchange</code>.<br><br>
<b>Why it matters</b><br>The Spark UI and query profile show stages. The plan shows operators. Mapping one to the other is how you find <b>which operators</b> are inside a slow stage.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The list of exchanges in the plan, and the resulting number of stages (exchanges + 1, plus one per broadcast).
</div>

In [0]:
lines = plan_text(query).splitlines()
exchanges = [l.strip() for l in lines if "Exchange" in l]
for e in exchanges:
    print(e[:110])
shuffles = sum(1 for e in exchanges if e.lstrip("+- ").startswith("Exchange"))
broadcasts = sum(1 for e in exchanges if "BroadcastExchange" in e)
print(f"shuffle exchanges: {shuffles} | broadcast exchanges: {broadcasts}")

## Under the hood

```
Physical plan (simplified, bottom-up)                         Stage
-------------------------------------------------------------------------
Sort revenue DESC                                             S4
  Exchange rangepartitioning(revenue)        ── shuffle ──
HashAggregate(final, country)                                 S3
  Exchange hashpartitioning(country)         ── shuffle ──
HashAggregate(partial, country)                               S2
  BroadcastHashJoin customer_id                               S2
    Filter year(order_date) = 2024                            S2
      FileScan parquet [customer_id, order_date, amount]      S2
    BroadcastExchange                         ── broadcast ── (S1 builds the small side)
      Range / customers                                       S1
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> each `Exchange` is a stage boundary in the DAG.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> everything between two exchanges runs in one stage, usually fused into one `*(n)` codegen block.

**AQE note:** `AdaptiveSparkPlan isFinalPlan=false` means this is the **initial** plan. At runtime, AQE can coalesce shuffle partitions, switch a sort-merge join to a broadcast join, or split skewed partitions. The final plan is visible in the Spark UI's SQL tab (classic compute) or the query profile (serverless). See lesson 13.

In [0]:
# Optional cleanup
dbutils.fs.rm("/Volumes/workspace/spark_basics/raw_files/lesson02_05", True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: reading the plan top-down and getting confused</b><br>
Data flows from the bottom (scans) to the top (result). Read bottom-up.<br><br>
**⛔ Problem: <code>PushedFilters: []</code> on a big scan**<br>The filter wraps the column in a function or UDF, or comes after an operation the optimizer can't push through. Rewrite it as a simple comparison on the raw column.<br><br>
**⛔ Problem: <code>CartesianProduct</code> or <code>BroadcastNestedLoopJoin</code> in the plan**<br>A missing or non-equi join condition. Check the join keys.<br><br>
<b>⛔ Problem: the plan in the notebook doesn't match what ran</b><br>With AQE, <code>explain</code> shows the initial plan. Check the final plan in the Spark UI or query profile.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What are the stages of query planning in Spark?</b><br>
Parsing produces an unresolved logical plan. The analyzer resolves names and types against the catalog. The Catalyst optimizer applies rule-based (and some cost-based) optimizations to produce an optimized logical plan. The planner turns that into a physical plan with concrete operators, and whole-stage code generation compiles parts of it. AQE can then re-optimize at runtime.<br><br>

<b>🎤 2. How do you read a physical plan?</b><br>
Bottom-up, from the scans to the result. I check the scans for <code>ReadSchema</code>, <code>PushedFilters</code> and <code>PartitionFilters</code>, count the <code>Exchange</code> operators to see the shuffles and stages, and check which join strategy was chosen.<br><br>

<b>🎤 3. What does <code>Exchange</code> mean in a plan?</b><br>
A shuffle: data is redistributed between partitions, for example <code>hashpartitioning</code> for a groupBy or join, or <code>rangepartitioning</code> for a sort. Each exchange is a stage boundary.<br><br>

<b>🎤 4. What does the <code>*(1)</code> prefix mean?</b><br>
The operator is part of whole-stage code generation stage 1: those operators are fused into one generated Java function that processes rows in a single loop.<br><br>

<b>🎤 5. What is the difference between the logical and the physical plan?</b><br>
The logical plan describes <b>what</b> to compute (relational operations). The physical plan describes <b>how</b>: which join algorithm, which aggregation strategy, where to shuffle.<br><br>

<b>🎤 6. What is a DAG in Spark?</b><br>
The directed acyclic graph of stages that the DAG scheduler builds from the physical plan, split at shuffle boundaries, with edges showing which stage's output feeds the next.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which command shows the parsed, analyzed, optimized and physical plans of a query?</b><br>
A. <code>df.explain()</code><br>
B. <code>df.explain(True)</code> (or <code>explain("extended")</code>)<br>
C. <code>df.printSchema()</code><br>
D. <code>DESCRIBE HISTORY</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>explain()</code> without arguments shows only the physical plan.</details><br><br>

<b>Q2. A physical plan contains <code>PushedFilters: []</code> for a large table scan although the query filters on <code>event_date</code>. What is a likely cause?</b><br>
A. The table is stored in Delta format<br>
B. The filter wraps <code>event_date</code> in a function or UDF, so it can't be pushed down<br>
C. The table has too few partitions<br>
D. Photon is disabled<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a query that joins <code>orders</code> to <code>customers</code>, keeps orders over 50, counts orders per country, and returns the top 2 countries</li><li>Print its formatted plan and list: the join strategy, the number of shuffle exchanges, and whether <code>amount &gt; 50</code> was pushed to the scan</li><li>Find the <code>TakeOrderedAndProject</code> operator and explain in one sentence why it appears</li><li>Change the query so that the customers side is forced to a sort-merge join, and count the exchanges again</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Top 2 countries
q = (
    orders.filter(F.col("amount") > 50)
    .join(customers, "customer_id")
    .groupBy("country").count()
    .orderBy(F.desc("count")).limit(2)
)
q.show()

# 2. Join strategy, exchanges, pushdown
text = plan_text(q, "formatted")
print("join:", "BroadcastHashJoin" if "BroadcastHashJoin" in text else "SortMergeJoin")
print("shuffle exchanges:", len(re.findall(r"\bExchange \(", text)))
print("amount pushed:", "GreaterThan(amount,50" in text)

# 3. orderBy + limit becomes a top-N: each partition keeps its best 2 rows, so no full sort is needed.
print("TakeOrderedAndProject present:", "TakeOrderedAndProject" in text)

# 4. Force sort-merge
q_smj = orders.filter(F.col("amount") > 50).join(customers.hint("merge"), "customer_id").groupBy("country").count()
print("exchanges with sort-merge:", len(re.findall(r"\bExchange \(", plan_text(q_smj, "formatted"))))

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Parsed → analyzed → optimized → physical plan, then a DAG of stages at runtime</li><li><code>explain()</code> physical only, <code>explain(True)</code> all phases, <code>explain("formatted")</code> best for reading</li><li>Read physical plans bottom-up. Check scans for <code>ReadSchema</code>, <code>PushedFilters</code>, <code>PartitionFilters</code></li><li><code>Exchange</code> = shuffle = stage boundary. <code>*(n)</code> = fused codegen stage</li><li>Join operators tell you the strategy: <code>BroadcastHashJoin</code>, <code>SortMergeJoin</code>, <code>CartesianProduct</code></li><li>With AQE, the final plan is in the Spark UI or the query profile</li></ul>
</div>

| Look for | Meaning |
|---|---|
| `ReadSchema` | Columns read (pruning) |
| `PushedFilters` / `DataFilters` | Filters given to the reader |
| `PartitionFilters` | Partition pruning |
| `Exchange` | Shuffle / new stage |
| `BroadcastHashJoin` / `SortMergeJoin` | Join strategy |
| `TakeOrderedAndProject` | Top-N |
| `AdaptiveSparkPlan` | AQE active |

## Git commit

```
git add 02_spark_internals/05_dag_and_query_plans.ipynb
git commit -m "add 02_05 dag and query plans notebook"
```